In [14]:
#!pip install --force-reinstall boto3
!pip install awscrt
#!pip install awsiotsdk

In [52]:
import boto3
import pandas as pd
import json
import os
import time
import logging
from datetime import datetime
from awscrt import mqtt
from awsiot import mqtt_connection_builder

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load CSV files
demand_df = pd.read_csv('./data/demand.csv')
demand_df = demand_df.head (5)

# AWS IoT Core settings
endpoint = "azdbv7nhy84sn-ats.iot.us-east-1.amazonaws.com"  # Get from IoT Core > Settings
cert_path = "./certificates/certificate.pem.crt"
pri_key_path = "./certificates/private.pem.key"
ca_path = "./certificates/AmazonRootCA1.pem"
client_id = "SimulatedSensor"
topic = "building/sensors"

# Create MQTT connection
mqtt_connection = mqtt_connection_builder.mtls_from_path(
    endpoint=endpoint,
    cert_filepath=cert_path,
    pri_key_filepath=pri_key_path,
    ca_filepath=ca_path,
    client_id=client_id
)

print(os.path.exists(cert_path))
print(os.path.exists(pri_key_path))
print(os.path.exists(ca_path))

import logging
logging.basicConfig(level=logging.DEBUG) 

print("Connecting to AWS IoT Core...")
connect_future = mqtt_connection.connect()
connect_future.result()
print("Connected!")

# Publish data
try:
    for index, row in demand_df.iterrows():
        message = {
            "timestamp": str(row['Timestamp']),
            "demand": float(row['Demand'])
        }
        
        mqtt_connection.publish(
            topic=topic,
            payload=json.dumps(message),
            qos=mqtt.QoS.AT_LEAST_ONCE
        )
        print(f"Published: {message}")
        time.sleep(1)  # Delay between messages

finally:
    disconnect_future = mqtt_connection.disconnect()
    disconnect_future.result()

True
True
True
Connecting to AWS IoT Core...
Connected!
Published: {'timestamp': '2015-01-01 05:00:00', 'demand': 210.9}
Published: {'timestamp': '2015-01-01 05:15:00', 'demand': 228.2}
Published: {'timestamp': '2015-01-01 05:30:00', 'demand': 231.1}
Published: {'timestamp': '2015-01-01 05:45:00', 'demand': 228.4}
Published: {'timestamp': '2015-01-01 06:00:00', 'demand': 231.8}
